# GRACE Projection V2 — benchmark ReVeal test

Nhánh `projection-v2-local-metric` dùng residual MLP + local contrastive training. Head đã train trên train và được chọn bằng validation retrieval proxy. Cấu hình hiện tại đã khóa để chạy benchmark **test** một lần; không điều chỉnh model/hyperparameter theo điểm test.

| Bước | Chạy ở đâu | Cần chuyển môi trường khi nào |
|---|---|---|
| 1. Kiểm tra head đã train và chọn bằng validation proxy | Local | Nếu thiếu head, mới cần local/Kaggle training |
| 2. Retrieval/graph/prompt cho ReVeal test | WSL local `/home` | Chạy `notebooks/local_cpu_prepare_v2.ipynb`; checkpoint/prepared zip xuất về `bundles/` trên ổ D |
| 3. Preflight → LLM → test evaluation | Local có Vertex ADC | Chỉ bật `RUN_INFERENCE=True` sau khi xem preflight |

Training đã dùng validation để chọn checkpoint. Notebook này chỉ chuẩn bị và đánh giá test, mặc định không gọi API mới. Kết quả test đã từng được xem trong các lượt H1 trước đây; xem giới hạn protocol trong `docs/PROJECTION_V2_RUNBOOK.md`.


In [1]:
from pathlib import Path
import os,sys,json,zipfile
candidates=([Path(os.environ['GRACE_H1_ROOT'])] if os.environ.get('GRACE_H1_ROOT') else [])
candidates += [Path.cwd(),*Path.cwd().parents,Path('D:/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection'),Path('/mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection')]
ROOT=next((p.resolve() for p in candidates if (p/'grace_h1').is_dir()),None)
if ROOT is None: raise RuntimeError('Set GRACE_H1_ROOT to the project folder')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
TRAIN_ON_KAGGLE=False
PREPARE_ON_LOCAL_NOTEBOOK=True  # WSL /home staging; kết quả được export về bundles/ local.
FORCE_FRESH_INFERENCE=True  # Bỏ qua toàn bộ response cache và prior-run archive.
RUN_INFERENCE=True   # True sẽ gọi Vertex cho prompt chưa có response cache.
from grace_h1.io import load_local_environment,file_hash
from grace_h1.experiment_v2 import load_config,validate_config,training_identity,require_heads,require_runs,train_all,prepare_all,preflight_all,infer_all,evaluate_all
from grace_h1.portable_v2 import create_bundle,import_bundle
load_local_environment(ROOT)
config=load_config(ROOT);validate_config(config)
HEADS=ROOT/'bundles/projection-v2-heads.zip'
PREPARED=ROOT/'bundles/projection-v2-cpu-prepared.zip'
print('Project:',ROOT,'\n',json.dumps(config,indent=2))


Project: /mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection 
 {
  "datasets": [
    "reveal"
  ],
  "splits": [
    "test"
  ],
  "objectives": [
    "residual_ce",
    "residual_local"
  ],
  "llm_arms": [
    "baseline_whitening",
    "h1_residual_local"
  ],
  "training": {
    "dimension": 256,
    "hidden_dimension": 256,
    "residual_radius": 0.25,
    "dropout": 0.1,
    "epochs": 30,
    "batch_size": 128,
    "learning_rate": 0.0003,
    "weight_decay": 0.001,
    "temperature": 0.1,
    "margin": 0.2,
    "local_weight": 0.1,
    "semi_hard_weight": 0.05,
    "geometry_weight": 1.0,
    "positive_neighbors": 8,
    "negative_neighbors": 32,
    "negatives_per_anchor": 8,
    "hard_warmup_epochs": 3,
    "hard_ramp_epochs": 5,
    "patience": 8,
    "seed": 42,
    "selection_min_gain": 0.002,
    "selection_max_class_drop": 0.02,
    "selection_min_teacher_overlap": 0.5,
    "device": "auto"
  },
  "pipeline": {
    "semantic_top_k": 5,
    "lexical_weight": 0.7,
    "whit

## Bước 1 — Kiểm tra head đã train

Head Projection V2 đã được train local. Cell dưới kiểm tra và nạp lại bundle hiện có; selection checkpoint vẫn dựa trên validation retrieval proxy. Bước này không train lại nếu head hợp lệ. Chỉ tạo training input và chuyển sang Kaggle/local train nếu thực sự thiếu head.


In [2]:
try:
    selection=require_heads(ROOT,config)
    print('Dùng lại trained heads local hợp lệ.')
except (FileNotFoundError,KeyError,ValueError) as local_exc:
    if HEADS.exists():
        print('Nạp trained heads từ bundle:',import_bundle(ROOT,HEADS,'heads'))
        selection=require_heads(ROOT,config)
    elif TRAIN_ON_KAGGLE:
        print(create_bundle(ROOT,ROOT/'bundles/projection-v2-training-inputs.zip','training_inputs'))
        raise RuntimeError('DỪNG BƯỚC 1: chạy notebooks/kaggle_train_projection_v2.ipynb trên Kaggle GPU; tải projection-v2-heads.zip về bundles/. '+str(local_exc)) from local_exc
    else:
        selection=train_all(ROOT,config)
for dataset,heads in selection['heads'].items():
    for objective,h in heads.items():
        print(dataset,objective,h['selection_status'],'best epoch',h['best']['epoch'],'macro P@5',h['best']['retrieval']['macro_label_precision_at_5'])


Dùng lại trained heads local hợp lệ.
reveal residual_ce validation_proxy_gain best epoch 4 macro P@5 0.6348957928946172
reveal residual_local validation_proxy_gain best epoch 2 macro P@5 0.6361136683784954


## Bước 2 — CPU preparation cho test

Config đã khóa `splits: ["test"]` và `test_protocol_frozen: true`. Bước này tạo retrieval, graph, rerank và prompt trên cohort test. Cache graph cũ được đóng gói và tái sử dụng theo code hash; chỉ graph còn thiếu mới phải parse.

Chạy `notebooks/local_cpu_prepare_v2.ipynb` bằng kernel WSL. Notebook nhập cache từ bundle local, xử lý trong `/home` rồi xuất `projection-v2-cpu-prepared.zip` và checkpoint về `bundles/` trên ổ D. Không cần Drive hay kết nối mạng ở bước CPU.

In [3]:
def cpu_inputs_match(path):
    try:
        with zipfile.ZipFile(path) as archive:
            job=json.loads(archive.read('bundle_manifest.json'))['job']
        if job['workflow']!='projection_v2' or job['experiment']!=config:
            return False
        if job['training_config_hash']!=training_identity(config) or job['data_manifest_sha256']!=file_hash(ROOT/'data/manifest.json'):
            return False
        return all(file_hash(ROOT/'grace_h1'/name)==digest for name,digest in job['source'].items())
    except (OSError,KeyError,ValueError,zipfile.BadZipFile):
        return False

try:
    run_rows=require_runs(ROOT,config)
    print('Dùng lại frozen V2 preparation local hợp lệ; không cần import bundle.')
except (FileNotFoundError,KeyError,ValueError) as local_exc:
    imported=False
    if PREPARED.exists():
        try:
            print('Nạp prepared bundle:',import_bundle(ROOT,PREPARED,'prepared'))
            run_rows=require_runs(ROOT,config)
            imported=True
        except (FileNotFoundError,KeyError,ValueError) as bundle_exc:
            print('Prepared bundle cũ/không khớp source; chuyển sang CPU preparation local:',bundle_exc)
    if not imported:
        if PREPARE_ON_LOCAL_NOTEBOOK:
            CPU_INPUTS=ROOT/'bundles/projection-v2-cpu-inputs.zip'
            if cpu_inputs_match(CPU_INPUTS):
                print('Dùng lại CPU input bundle đúng config; graph cache đã đóng gói sẵn:',CPU_INPUTS)
            else:
                print(create_bundle(ROOT,CPU_INPUTS,'cpu_inputs'))
            raise RuntimeError('DỪNG BƯỚC 2: chạy notebooks/local_cpu_prepare_v2.ipynb bằng WSL; prepared bundle sẽ được xuất thẳng về bundles/. '+str(local_exc)) from local_exc
        run_rows=prepare_all(ROOT,config)
print(run_rows)


Dùng lại frozen V2 preparation local hợp lệ; không cần import bundle.
[{'arm': 'baseline_whitening', 'dataset': 'reveal', 'path': 'runs/reveal/test/baseline_whitening/2d1845eabf62eb84', 'split': 'test'}, {'arm': 'h1_residual_local', 'dataset': 'reveal', 'path': 'runs/reveal/test/h1_residual_local/98d7f32cd098d4c0', 'split': 'test'}]


## Bước 3 — Preflight và inference local

Cell preflight không gọi API. `FORCE_FRESH_INFERENCE=True` buộc gửi request mới cho mọi prompt unique ở mỗi arm, bỏ qua cache cũ và response vừa sinh từ arm trước. `RUN_INFERENCE=True` sẽ chạy Vertex bằng ADC local; lượt này dự kiến gọi khoảng 2.270 prompt unique cho mỗi arm. Hai arm dùng chung prompt template/graph/query cohort/generation settings.

In [4]:
import importlib
import grace_h1.inference as inference_module
import grace_h1.experiment_v2 as experiment_v2_module
importlib.reload(inference_module)
importlib.reload(experiment_v2_module)
print('Inference code:',experiment_v2_module.__file__)
plans=experiment_v2_module.preflight_all(ROOT,config,force_fresh=FORCE_FRESH_INFERENCE)
for plan in plans: print(plan)


Inference code: /mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection/grace_h1/experiment_v2.py
{'arm': 'baseline_whitening', 'dataset': 'reveal', 'path': 'runs/reveal/test/baseline_whitening/2d1845eabf62eb84', 'split': 'test', 'queries': 2274, 'unique_prompts': 2270, 'cached_queries': 0, 'new_unique_requests': 2270, 'imported_prior_responses': 0, 'request_spec': {'model': 'publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas', 'provider': 'Vertex AI', 'inference': {'temperature': 0.0, 'seed': 42, 'max_tokens': 256, 'retry_policy': 'max_attempts=5; exponential_backoff=5s..120s; exhausted_request=invalid', 'parser': '../Grace-with-filters-v2/[CONTRACT]/parser/parse_prediction.py::parse_prediction', 'parser_contract': 'exact_last_line_binary_v1'}}, 'force_fresh': True, 'worst_case_attempts': 11350}
{'arm': 'h1_residual_local', 'dataset': 'reveal', 'path': 'runs/reveal/test/h1_residual_local/98d7f32cd098d4c0', 'split': 'test', 'queries': 2274, 'unique_prompts': 2270, 'cached_queries'

In [5]:
if RUN_INFERENCE:
    inference=experiment_v2_module.infer_all(ROOT,config,allow_api=True,force_fresh=FORCE_FRESH_INFERENCE)
    for item in inference: print(item)
else:
    print('Chưa gọi API. Đặt RUN_INFERENCE=True sau khi xem preflight nếu muốn chạy inference.')


LLM unique requests 25/2270
LLM unique requests 50/2270
LLM unique requests 75/2270
LLM unique requests 100/2270
LLM unique requests 125/2270
LLM unique requests 150/2270
LLM unique requests 175/2270
LLM unique requests 200/2270
LLM unique requests 225/2270
LLM unique requests 250/2270
LLM unique requests 275/2270
LLM unique requests 300/2270
LLM unique requests 325/2270
LLM unique requests 350/2270
LLM unique requests 375/2270
LLM unique requests 400/2270
LLM unique requests 425/2270
LLM unique requests 450/2270
LLM unique requests 475/2270
LLM unique requests 500/2270
LLM unique requests 525/2270
LLM unique requests 550/2270
LLM unique requests 575/2270
LLM unique requests 600/2270
LLM unique requests 625/2270
LLM unique requests 650/2270
LLM unique requests 675/2270
LLM unique requests 700/2270
LLM unique requests 725/2270
LLM unique requests 750/2270
LLM unique requests 775/2270
LLM unique requests 800/2270
LLM unique requests 825/2270
LLM unique requests 850/2270
LLM unique reques

In [7]:
# Retry riêng các request exhausted trong run hiện tại; request thành công không bị gọi lại.
# Chạy cell khởi tạo (cell 1) trước nếu kernel đã restart.
import importlib
import grace_h1.inference as inference_module
import grace_h1.experiment_v2 as experiment_v2_module
importlib.reload(inference_module)
importlib.reload(experiment_v2_module)
from grace_h1.io import read_json
run_config=read_json(ROOT/'runs/projection_v2/run_selection.json')['experiment_config']
retry_workers=read_json(ROOT/'configs/projection_v2.json')['inference_workers']
print('Retry workers:',retry_workers)
retry_reports=experiment_v2_module.retry_exhausted_all(ROOT,run_config,workers=retry_workers)
for item in retry_reports: print(item)


Retry workers: 5
Retry exhausted only: reveal/test/baseline_whitening with 5 workers
LLM unique requests 25/148
LLM unique requests 50/148
LLM unique requests 75/148
LLM unique requests 100/148
LLM unique requests 125/148
LLM unique requests 148/148
Retry exhausted only: reveal/test/h1_residual_local with 5 workers
LLM unique requests 25/128
LLM unique requests 50/128
LLM unique requests 75/128
LLM unique requests 100/128
LLM unique requests 125/128
LLM unique requests 128/128
{'arm': 'baseline_whitening', 'dataset': 'reveal', 'path': 'runs/reveal/test/baseline_whitening/2d1845eabf62eb84', 'split': 'test', 'queries': 2274, 'unique_prompts': 2270, 'cached_queries': 2125, 'new_unique_requests': 148, 'imported_prior_responses': 0, 'request_spec': {'model': 'publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas', 'provider': 'Vertex AI', 'inference': {'temperature': 0.0, 'seed': 42, 'max_tokens': 256, 'retry_policy': 'max_attempts=5; exponential_backoff=5s..120s; exhausted_request=inv

## Bước 4 — Paired test evaluation

Báo cáo V2 nằm trong `runs/projection_v2/experiment_summary.json`; retrieval proxy không thay thế F1/MCC của GRACE. Config test đã khóa. Chạy preflight, sau đó đặt `RUN_INFERENCE=True` để gọi Vertex bằng ADC local nếu còn request thiếu. Không dùng kết quả test để chọn seed, epoch, model hay hyperparameter.


In [8]:
if all((ROOT/r['path']/'predictions.jsonl').exists() for r in run_rows):
    report=evaluate_all(ROOT,config)
    print('Status:',report['status'])
    for row in report['scores']: print(row['dataset'],row['arm'],row['metrics'])
    for comparison in report['comparisons']: print(comparison)
else:
    print('Chưa có đủ predictions; tiếp tục Bước 3 trước khi evaluation.')


Status: partial
reveal baseline_whitening {'n': 2274, 'valid_n': 2262, 'invalid_n': 12, 'invalid_rate': 0.005277044854881266, 'confusion_matrix_valid_only': {'tp': 73, 'tn': 1487, 'fp': 546, 'fn': 156}, 'precision': 0.11793214862681745, 'recall': 0.31877729257641924, 'f1': 0.17216981132075473, 'mcc': 0.03397043301883586, 'balanced_accuracy': 0.5251043373851108, 'accuracy_valid_only': 0.6896551724137931, 'accuracy_all': 0.6860158311345647, 'recall_all_positives': 0.3173913043478261, 'predicted_positive_rate_all': 0.272207563764292, 'invalid_positive_n': 1, 'metric_scope': 'binary metrics condition on valid labels; invalids retained and reported separately', 'trivial_baselines': {'always_vulnerable': {'n': 2274, 'valid_n': 2274, 'invalid_n': 0, 'invalid_rate': 0.0, 'confusion_matrix_valid_only': {'tp': 230, 'tn': 0, 'fp': 2044, 'fn': 0}, 'precision': 0.1011433597185576, 'recall': 1.0, 'f1': 0.18370607028753994, 'mcc': 0.0, 'balanced_accuracy': 0.5, 'accuracy_valid_only': 0.10114335971855